In [0]:
df_silver = spark.table("workspace.gov.silver_bolsa_atleta")

# criei dimensao
dim_atleta = df_silver.select("cpf", "nome_atleta", "municipio", "uf").dropDuplicates(["cpf"])
dim_atleta.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gov.dim_atleta")

# criei fato
fato_pagamento = df_silver.select("cpf", "edital", "categoria", "modalidade", "situacao", "valor_pago", "data_pagamento", "data_referencia")
fato_pagamento.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gov.fato_pagamento")

print("Camada Gold criada com sucesso: dim_atleta e fato_pagamento")

display(dim_atleta)

Camada Gold criada com sucesso: dim_atleta e fato_pagamento


cpf,edital,categoria,modalidade,situacao,valor_pago,data_pagamento,data_referencia
***.076.617-**,2025,Atleta Nacional,Judô,Pago,1025.0,2026-04-30,2026/04
***.821.148-**,2025,Atleta Nacional,Futebol,Pago,1025.0,2026-04-30,2026/04
***.635.021-**,2026,Atleta Nacional,Judô de Cegos,Pago,1025.0,2026-04-30,2026/04
***.237.368-**,2025,Atleta Nacional,Tênis de Mesa,Pago,1025.0,2026-04-30,2026/04
***.600.051-**,2026,Atleta Nacional,Flag Futebol,Pago,1025.0,2026-04-30,2026/04
***.393.114-**,2025,Atleta Nacional,Atletismo Paralímpico,Pago,1025.0,2026-04-30,2026/04
***.504.806-**,2026,Atleta Nacional,Vôlei de Quadra,Pago,1025.0,2026-04-30,2026/04
***.362.786-**,2025,Atleta Nacional,Taekwondo (Kyorugi),Pago,1025.0,2026-04-30,2026/04
***.995.717-**,2025,Atleta Nacional,Tiro com Arco,Pago,1025.0,2026-04-30,2026/04
***.987.401-**,2025,Atleta Nacional,Hóquei sobre a Grama,Pago,1025.0,2026-04-30,2026/04


In [0]:
%sql
alter table workspace.gov.fato_pagamento drop constraint if exists pagamento_fk;
alter table workspace.gov.dim_atleta drop constraint if exists atleta_pk;

--  alterar a coluna cpf para NOT NULL
alter table workspace.gov.dim_atleta alter column cpf set not null;

--  adicionar a PRIMARY KEY
alter table workspace.gov.dim_atleta add constraint atleta_pk primary key (cpf);

-- a FOREIGN KEY
alter table workspace.gov.fato_pagamento add constraint pagamento_fk foreign key (cpf)
references workspace.gov.dim_atleta(cpf);